In [1]:
from pathlib import Path
import xarray as xr
import warnings

In [13]:
import xarray as xr
from pathlib import Path
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Directorio donde están los originales
directorio = Path(".")
salida = Path("figuras")
salida.mkdir(exist_ok=True)

temp_vars = [
    "ctd_temperature",
    "ctd_temperature_68",
    "ctd_temperature_alt_1",
    "ctd_temperature_unk",
    "potential_temperature",
    "potential_temperature_c",
]

with open("A0sin_clasificar.txt") as f:
    for nombre in map(str.strip, f):

        if not nombre:
            continue

        encontrados = list(directorio.glob(f"*_{nombre}"))

        if not encontrados:
            continue

        fichero = encontrados[0]

        try:
            ds = xr.open_dataset(fichero, decode_timedelta=False)

            # Buscar una variable de temperatura
            nestaciones = None
            for var in temp_vars:
                if var in ds.variables:
                    nestaciones = ds[var].shape[0]
                    break

            if nestaciones is None or nestaciones <= 20:
                ds.close()
                continue

            if "longitude" not in ds or "latitude" not in ds:
                print(f"{fichero.name}: sin longitude/latitude")
                ds.close()
                continue

            lon = ds["longitude"].values
            lat = ds["latitude"].values

            fig = plt.figure(figsize=(12,6))
            ax = plt.axes(projection=ccrs.PlateCarree())

            ax.set_global()

            ax.add_feature(cfeature.LAND, facecolor="lightgray")
            ax.add_feature(cfeature.OCEAN, facecolor="white")
            ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
            ax.add_feature(cfeature.BORDERS, linewidth=0.3)

            gl = ax.gridlines(draw_labels=True, linewidth=0.3)
            gl.top_labels = False
            gl.right_labels = False

            ax.plot(
                lon,
                lat,
                "-o",
                color="red",
                markersize=3,
                linewidth=1,
                transform=ccrs.PlateCarree(),
            )

            ax.set_title(f"{fichero.stem} ({nestaciones} estaciones)")

            plt.savefig(
                salida / f"{fichero.stem}.png",
                dpi=200,
                bbox_inches="tight",
            )
            plt.close(fig)

            ds.close()

        except Exception as e:
            print(f"ERROR: {fichero.name}: {e}")